# Intro to wavelets

### Why wavelets?

MRI acquires data in **k-space** — the Fourier domain of the image. Compressed sensing (CS) promises faithful reconstruction from far fewer k-space samples than the Nyquist limit requires, but only if the image has a **sparse representation** in some transform domain. Wavelets provide exactly that: most natural images, including MR images, concentrate their energy in a small number of wavelet coefficients. Understanding *why* wavelets produce sparsity is the goal of this chapter — and the foundation for everything that follows on compressed sensing.

---

**Approach.** This activity follows a discovery-based pedagogy. You will start with a kernel that cannot possibly work, identify why it fails, fix its deficiencies one by one, and arrive at a working wavelet transform — not by memorising axioms, but by deriving each property from necessity.

**Signal design.** The signal you will build contains two frequency components inside a smooth window. By adjusting the amplitudes, you can make one component dominate the other visually. As you progress through the sections, you will watch the wavelet transform *reveal* hidden components, separate them into different scales, remove the dominant one to expose the quiet one, and compress by discarding near-zero coefficients.

In [1]:
# Helper to load JS files and build HTML with Plotly wait pattern
import pathlib
from IPython.display import HTML

_root = pathlib.Path.cwd()
if not (_root / '_static').exists():
    _root = _root.parent

def _read_js(*names):
    return '\n'.join((_root / '_static' / 'js' / n).read_text() for n in names)

def _embed(controls_html, js_code, wait_for='WaveletLib'):
    """Wrap HTML controls + JS in the Plotly loader pattern."""
    if wait_for:
        init_guard = f'''if (typeof window.{wait_for} === "undefined") {{ setTimeout(initBlock, 100); return; }}'''
    else:
        init_guard = ''
    return HTML(f'''
<style>
.controls-panel {{
  background: #f9f6f2; border: 1px solid #e0dcd6; border-radius: 4px;
  padding: 1rem 1.2rem; margin: 1.5rem 0; display: flex; flex-wrap: wrap;
  gap: 0.8rem 1.5rem; align-items: end;
}}
.controls-panel label {{
  display: flex; flex-direction: column; font-size: 0.85rem; color: #555; gap: 0.2rem;
}}
.controls-panel input[type="number"], .controls-panel select {{
  font-size: 0.9rem; padding: 0.25rem 0.4rem; border: 1px solid #ccc;
  border-radius: 3px; width: 7rem;
}}
.controls-panel select {{ width: 8rem; }}
</style>
{controls_html}
<script>
(function() {{
  function initBlock() {{
    {init_guard}
    {js_code}
  }}
  if (typeof Plotly !== "undefined") {{ initBlock(); }}
  else {{ setTimeout(initBlock, 200); }}
}})();
</script>
''')

---

## 1. Build your signal

A) Start by defining a signal $h(x)$ of the form:

$$h(x) = w(x,\,\sigma) \cdot \bigl[a_1\sin(\omega_1 x) + a_2\cos(\omega_2 x)\bigr]$$

where $w(x, \sigma)$ is a smooth window function (e.g. a Gaussian or Fermi function) with a width parameter $\sigma$, $\omega_1$ is a low frequency, and $\omega_2$ is a high frequency. Try setting the amplitudes so that one component dominates the other (e.g. $a_1 = 1, a_2 = 10$) — the weaker oscillation should be nearly invisible in the raw signal but clearly visible in the wavelet coefficients at the appropriate scale. Using sin for one component and cos for the other ensures that $h(x)$ is neither symmetric nor anti-symmetric, so no symmetry shortcuts are available. Choose $\omega_1$ and $\omega_2$ sufficiently far apart. Ensure that $h(x)$ tends rapidly to zero for large $|x|$.

B) Plot $h(x)$ below. If you've set the amplitudes so that one frequency dominates, notice whether you can see the weaker oscillation at all — it should be nearly buried. This is deliberate: you will later use the wavelet transform to reveal it.

C) Try adjusting $\sigma$: make it wide enough that several full periods of $\omega_1$ fit inside the window, then try making it too narrow — so narrow that fewer than one or two periods of $\omega_1$ fit inside. Plot both cases. In the narrow case, can you distinguish the slow oscillation from a simple slope or offset? Keep the wider $\sigma$ for all subsequent sections.

> *Your signal is the unknown you want to decompose. The kernels you will try in the following sections play the same role that cos, sin, and complex exponentials play in Fourier analysis. The window width matters practically: if $\sigma$ is too small, the low frequency doesn't have room to express itself, and no transform can recover what isn't there.*

In [2]:
# Q1: signal builder — loads Plotly + signal.js + q1a.js
signal_js = _read_js('signal.js')
q1a_js = _read_js('q1a.js')

q1_controls = '''
<div class="controls-panel">
  <label>Window <select id="q1a-window-type">
    <option value="gaussian">Gaussian</option><option value="fermi">Fermi</option>
  </select></label>
  <span id="q1a-gaussian-params" style="display:contents">
    <label>\u03c3 <input type="number" id="q1a-sigma" step="0.1" min="0.1"></label>
  </span>
  <span id="q1a-fermi-params" style="display:none">
    <label>width <input type="number" id="q1a-fermi-width" step="0.5" min="0.5"></label>
    <label>edge <input type="number" id="q1a-fermi-edge" step="0.05" min="0.01"></label>
  </span>
  <label>center <input type="number" id="q1a-center" step="0.5"></label>
  <label>\u03c9\u2081 <input type="number" id="q1a-omega1" step="0.5" min="0.1"></label>
  <label>\u03c9\u2082 <input type="number" id="q1a-omega2" step="1" min="0.1"></label>
  <label>amp\u2081 <input type="number" id="q1a-amp1" step="0.5"></label>
  <label>amp\u2082 <input type="number" id="q1a-amp2" step="1"></label>
  <label>t_max <input type="number" id="q1a-tmax" step="1" min="1"></label>
  <label>N <input type="number" id="q1a-N" step="128" min="128"></label>
</div>
<div id="q1a-plot" style="width:100%;height:400px;"></div>
'''

# Q1 is special: it loads Plotly and signal.js (no wait_for needed)
HTML(f'''
<style>
.controls-panel {{
  background: #f9f6f2; border: 1px solid #e0dcd6; border-radius: 4px;
  padding: 1rem 1.2rem; margin: 1.5rem 0; display: flex; flex-wrap: wrap;
  gap: 0.8rem 1.5rem; align-items: end;
}}
.controls-panel label {{
  display: flex; flex-direction: column; font-size: 0.85rem; color: #555; gap: 0.2rem;
}}
.controls-panel input[type="number"], .controls-panel select {{
  font-size: 0.9rem; padding: 0.25rem 0.4rem; border: 1px solid #ccc;
  border-radius: 3px; width: 7rem;
}}
.controls-panel select {{ width: 8rem; }}
</style>
{q1_controls}
<script>
(function() {{
  function initAll() {{
    {signal_js}
    {q1a_js}
  }}
  if (typeof Plotly !== "undefined") {{ initAll(); }}
  else {{
    var s = document.createElement("script");
    s.src = "https://cdn.plot.ly/plotly-2.35.2.min.js";
    s.onload = initAll;
    document.head.appendChild(s);
  }}
}})();
</script>
''')

---

## 2. Try a positive kernel

Using your signal from above (with the wider $\sigma$):

A) Choose a smooth, positive kernel $\varphi(x)$ (for example a Gaussian) with a width parameter. Your kernel should be 32 samples long (with $N = 1024$, this ensures the kernel fits comfortably at every level of a 4-level decomposition, where the coarsest array has $N/8 = 128$ samples). Convolve it with your signal $h(x)$ at three different kernel widths. Plot the convolution output in each case.

B) Examine the convolution output. Can you detect the oscillatory structure of $h(x)$? Can you distinguish the two frequencies? Think about what the convolution is actually computing and why the output behaves this way.

C) Consider the fundamental limitation of using a kernel with nonzero mean. What property must a kernel have to detect changes rather than just smooth?

> *This parallels the Fourier world: cosines alone cannot capture the odd part of a signal. A positive kernel is structurally incapable of seeing the detail, just as cosines alone cannot capture the odd part.*

In [3]:
# Q2: positive kernel plot + convolution visualizer
_embed(
    '''
    <div class="controls-panel">
      <label>Kernel length <input type="number" id="q2a-kernel-length" step="2" min="4"></label>
      <label>\u03c3 <input type="number" id="q2a-sigma" step="0.5" min="0.5"></label>
    </div>
    <div id="q2a-plot" style="width:100%;height:300px;"></div>
    <div style="margin:1.5rem 0;"><label style="font-size:0.85rem;color:#555;">Kernel position
      <input type="range" id="q2conv-slider" style="width:100%;margin-top:0.3rem;">
    </label></div>
    <div id="q2conv-plot" style="width:100%;height:500px;"></div>
    ''',
    _read_js('q2a.js', 'q2conv.js')
)

---

## 3. Try a rough, zero-mean kernel

A) Construct a zero-mean kernel: a function whose integral is zero. Make it deliberately rough — a sawtooth with positive and negative lobes, a clipped ramp, a triangular wave, or anything discontinuous. Convolve it with $h(x)$ at several widths.

B) Compare the output with the positive-kernel result from Section 2. Can you now detect changes in the signal? Can you see evidence of the oscillatory structure?

C) Examine the output carefully. Do you see spurious ringing, ripples, or artifacts that are not present in the original signal? If so, think about where they come from in terms of the kernel's frequency response. What property would the kernel need to eliminate these artifacts?

> *You have now discovered two of the core wavelet requirements: zero mean ($\int\psi = 0$) and smoothness. Each was motivated by a failure.*

In [4]:
# Q3: zero-mean kernel + convolution
_embed(
    '''
    <div class="controls-panel">
      <label>Kernel type <select id="q3a-kernel-type">
        <option value="bipolar">Bipolar Gaussian</option>
        <option value="dog">Difference of Gaussians</option>
      </select></label>
      <label>Length <input type="number" id="q3a-kernel-length" step="2" min="4"></label>
      <span id="q3a-bipolar-params" style="display:contents">
        <label>\u03c3 <input type="number" id="q3a-bipolar-sigma" step="0.5" min="0.5"></label>
        <label>separation <input type="number" id="q3a-bipolar-sep" step="1" min="1"></label>
      </span>
      <span id="q3a-dog-params" style="display:none">
        <label>\u03c3\u2081 (narrow) <input type="number" id="q3a-dog-sigma1" step="0.5" min="0.5"></label>
        <label>\u03c3\u2082 (wide) <input type="number" id="q3a-dog-sigma2" step="0.5" min="1.0"></label>
      </span>
    </div>
    <div id="q3a-plot" style="width:100%;height:300px;"></div>
    <div style="margin:1.5rem 0;"><label style="font-size:0.85rem;color:#555;">Kernel position
      <input type="range" id="q3conv-slider" style="width:100%;margin-top:0.3rem;">
    </label></div>
    <div id="q3conv-plot" style="width:100%;height:500px;"></div>
    ''',
    _read_js('q3a.js', 'q3conv.js')
)

---

## 4. The Haar wavelet at a single scale

The Haar wavelet is the simplest function that satisfies the zero-mean condition:

$$\psi(x) = \begin{cases} +1 & 0 \le x < \tfrac{1}{2} \\ -1 & \tfrac{1}{2} \le x < 1 \\ 0 & \text{otherwise} \end{cases}$$

A) Convolve $h(x)$ with the Haar wavelet at a scale matched to the high frequency $\omega_2$ (i.e., set the wavelet width to approximately one period of the fast oscillation). Plot the output. Which frequency component can you see in the coefficients?

B) Now set the scale to match the low frequency $\omega_1$ (wavelet width $\approx$ one period of the slow oscillation). Plot the output. Which frequency component dominates?

C) Notice that a single, fixed-scale wavelet cannot capture both frequency components simultaneously. Think about why, and what this implies about the requirements for a complete decomposition.

> *This parallels the Fourier case: just as cosines alone cannot capture the odd part of a signal, a wavelet at one scale cannot capture features at a different scale. The restriction is parity in Fourier, scale in wavelets.*

In [5]:
# Q4: Haar wavelet + convolution
_embed(
    '''
    <div class="controls-panel">
      <label>Haar width (samples) <input type="number" id="q4a-haar-width" step="2" min="2"></label>
    </div>
    <div id="q4a-plot" style="width:100%;height:300px;"></div>
    <div style="margin:1.5rem 0;"><label style="font-size:0.85rem;color:#555;">Kernel position
      <input type="range" id="q4conv-slider" style="width:100%;margin-top:0.3rem;">
    </label></div>
    <div id="q4conv-plot" style="width:100%;height:500px;"></div>
    ''',
    _read_js('q4a.js', 'q4conv.js')
)

---

## 5. The filter bank: applying both filters

Instead of convolving with the wavelet at different widths directly, the discrete wavelet transform uses a pair of short convolution filters derived from the wavelet:

- **Low-pass filter (from the scaling function $\varphi$):** captures the smooth, slowly-varying part.
- **High-pass filter (from the wavelet $\psi$):** captures the fast-changing detail.

For the Haar wavelet, these are simply $h = [1/\sqrt{2},\; 1/\sqrt{2}]$ (low-pass) and $g = [1/\sqrt{2},\; -1/\sqrt{2}]$ (high-pass).

A) Discretise your signal $h(x)$ into a vector of $N$ samples (choose $N$ to be a power of 2, e.g. 512 or 1024). Convolve with both filters and downsample by 2. You now have two output vectors of length $N/2$. Call them $a_1$ (from the low-pass) and $d_1$ (from the high-pass). Plot both.

B) Observe what $a_1$ looks like compared to the original signal, and what $d_1$ looks like. The original signal is dominated by the high-frequency component ($\omega_2$). Notice whether you can now see the quiet low-frequency component ($\omega_1$) emerging in $a_1$, even though it was nearly invisible in $h(x)$.

C) Think about why you need both $a_1$ and $d_1$ to represent the original signal. What information is in $a_1$ that is absent from $d_1$, and vice versa? This parallels the Fourier result that you need both cosines and sines for a complete basis.

> *The scaling function $\varphi$ and the wavelet $\psi$ are complementary, just as cos and sin are complementary. $\varphi$ is "blind" to detail (like cos is blind to the odd part), and $\psi$ is blind to the overall level (like sin is blind to the even part). Together they span the full space.*

In [6]:
# Q5: filter bank — high-pass and low-pass
_embed(
    '''
    <div class="controls-panel">
      <label>Filter length <input type="number" id="q5a-filter-length" value="32" step="2" min="2"></label>
    </div>
    <div style="margin:1.5rem 0;"><label style="font-size:0.85rem;color:#555;">High-pass filter position
      <input type="range" id="q5a-hi-slider" style="width:100%;margin-top:0.3rem;">
    </label></div>
    <div id="q5a-hi-plot" style="width:100%;height:450px;"></div>
    <div style="margin:1.5rem 0;"><label style="font-size:0.85rem;color:#555;">Low-pass filter position
      <input type="range" id="q5a-lo-slider" style="width:100%;margin-top:0.3rem;">
    </label></div>
    <div id="q5a-lo-plot" style="width:100%;height:450px;"></div>
    ''',
    _read_js('q5a.js')
)

---

## 6. The cascade: multi-scale decomposition

A) Take $a_1$ (the low-pass output from Section 5) and apply the same two filters again, followed by downsampling by 2. You now have $a_2$ (length $N/4$) and $d_2$ (length $N/4$). Repeat once more to get $a_3$ (length $N/8$) and $d_3$ (length $N/8$).

B) Plot $d_1$, $d_2$, $d_3$, and $a_3$ stacked vertically, aligned by position in the signal. At which level do the high-frequency oscillations ($\omega_2$) appear? At which level do the low-frequency oscillations ($\omega_1$) appear? What does $a_3$ contain?

C) Verify that the total number of coefficients stored ($|d_1| + |d_2| + |d_3| + |a_3|$) equals $N$, the original number of samples. Think about why this is the case and why it matters for compression.

D) Why does each level operate on the previous level's low-pass output rather than on the original signal? What would happen (in terms of redundancy and total number of coefficients) if every level operated on the original instead?

> *This is the central structural insight. The cascade gives you a non-redundant decomposition. Operating on the original at every scale (the CWT approach) is valid but redundant — you get more coefficients than input samples and cannot compress.*

In [7]:
# Q6: cascade decomposition
_embed(
    '''
    <div class="controls-panel">
      <label>Decomposition levels <input type="number" id="q6a-levels" value="3" step="1" min="1" max="8"></label>
    </div>
    <div id="q6a-plot" style="width:100%;height:600px;"></div>
    <div id="q6b-plot" style="width:100%;height:600px;"></div>
    <p style="font-size:0.85rem;color:#555;margin-top:0.3rem;">Total coefficients: <span id="q6b-count"></span></p>
    ''',
    _read_js('q6.js')
)

---

## 7. Reconstruction from partial information

> **How reconstruction works.** The decomposition at each level was: convolve with a filter, then downsample by 2. Reconstruction reverses both operations at each level, working from the coarsest level back to the finest.
>
> To reconstruct $a_{k-1}$ from $a_k$ and $d_k$:
> 1. **Upsample** $a_k$ by 2: insert a zero after every sample, e.g. $[x, y, z] \to [x, 0, y, 0, z, 0]$.
> 2. **Convolve** the upsampled $a_k$ with the synthesis low-pass filter $\tilde{h}$.
> 3. **Upsample** $d_k$ by 2 in the same way.
> 4. **Convolve** the upsampled $d_k$ with the synthesis high-pass filter $\tilde{g}$.
> 5. **Add** the two results element-wise: $a_{k-1} = (\tilde{h} * (\uparrow 2\, a_k)) + (\tilde{g} * (\uparrow 2\, d_k))$.
>
> For the Haar wavelet the synthesis filters are $\tilde{h} = [1/\sqrt{2},\; 1/\sqrt{2}]$ and $\tilde{g} = [-1/\sqrt{2},\; 1/\sqrt{2}]$ (the time-reversed analysis filters). For a 3-level decomposition you repeat this three times: $a_3 + d_3 \to a_2$, then $a_2 + d_2 \to a_1$, then $a_1 + d_1 \to h(t)$.

Using the checkboxes below, try reconstructing the signal keeping only one set of coefficients at a time. Try each in turn: keep only $d_3$ (128 coefficients), then only $d_2$ (256 coefficients), then only $d_1$ (512 coefficients), then only $a_3$ (128 coefficients). Each reconstruction isolates the content of one level. Compare each to the original signal (1024 samples). Note that storing all four sets of coefficients ($d_1 + d_2 + d_3 + a_3$) also requires 1024 values — the decomposition is non-redundant. At which level does the high-frequency component ($\omega_2$) live? At which level does the low-frequency component ($\omega_1$) appear? Which combination of levels would you keep to reveal the hidden slow oscillation while discarding the dominant fast one?

> *Each detail level captures a different frequency band. By selectively keeping or removing levels, you can isolate or suppress specific frequency components — and the memory cost of each choice is visible in the coefficient count.*

In [8]:
# Q7: reconstruction with checkboxes
_embed(
    '''
    <div class="controls-panel" id="q7a-checkboxes" style="gap:0.5rem 1.5rem;"></div>
    <div id="q7a-plot" style="width:100%;height:700px;"></div>
    <div id="q7a-memory" style="margin-top:0.5rem;"></div>
    ''',
    _read_js('q7a.js')
)

---

## 8. Experiment with your signal

Go back to Section 1 and change the parameters of your signal — try different frequencies, amplitudes, window widths, and window types. Then return to the decomposition (Section 6) and reconstruction (Section 7) and observe how the wavelet coefficients and compression behaviour change.

A) Find a set of parameters in Section 1 for which keeping only $a_3$ in Section 7 does *not* reconstruct the signal well. Think about why — what property of the signal causes the coarsest approximation to miss important information?

B) What happens when you make the window narrower (so that fewer periods of $\omega_1$ fit inside)? Can the wavelet transform still separate the two frequencies? Can you still recover the low-frequency component by keeping only certain levels?

C) What happens when you bring $\omega_1$ and $\omega_2$ closer together? At what point do the two components start to overlap in the same detail levels?

> *The point of this section is to develop intuition for what the wavelet transform can and cannot do. The default parameters were designed to make the separation dramatic — but real signals are rarely so cooperative.*

---

## 9. The wavelet transform at a glance

The four plots below summarise the entire wavelet pipeline. The top plot shows the original signal. The second plot shows all wavelet coefficients laid out in a single row at the original signal's length — each colour represents a different decomposition level. The third plot shows the same coefficient layout but with everything zeroed except $a_3$. The bottom plot shows the signal reconstructed from only $a_3$.

Use the level selector from Section 6 and the signal controls from Section 1 to explore. Which colour bands carry the most energy? Which ones can you zero out with minimal reconstruction error?

In [9]:
# Q9: wavelet transform summary
_embed(
    '<div id="q9-plot" style="width:100%;height:700px;"></div>',
    _read_js('q9.js')
)

The next chapter extends this decomposition to two dimensions, applying the same filter-bank logic to images — row by row, then column by column.

You now have the core machinery: a non-redundant, multi-scale decomposition that concentrates signal energy into a small number of large coefficients. That concentration is exactly the **sparsity** that compressed sensing exploits to reconstruct MR images from far fewer k-space samples than conventional sampling requires.